In [2]:
import math
import time
import pandas as pd
import numpy as np 
from numpy import array
from matplotlib import pyplot
from tensorflow import keras 
from keras.models import Sequential 
from keras.layers import LSTM 
from keras.layers import Dense, Dropout  
from keras.models import Model 
from keras.layers import Input
from keras.layers import concatenate
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

In [13]:
# def main():
   
#     n_steps_in =12  # input y sequence for LSTM cell
#     n_features = 1  # one feature for the input of the LSTM cell
#     n_steps_out =6# num of predicted steps. 
#     n_epoch_global=15
#     n_trivals=10
#     n_out=5  
#     n_n_lstm=36
#     dropout=0.2
#     bat_size= 30
#     accuracy_avg_1=[]
#     accuracy_avg_2=[]
#     flag_sensitivity=0  
#     model_id='Mix_LSTM' 
#     flag_ML=0 # to run machine learning models, set  flag_ML=1 otherwise 0
#     # for ML, we set n_steps_out=36 as we compute the predcition for all forecasting cases 
#     if  flag_ML==1:
#         n_steps_out=36
#     #choose the ML model to test
    
#     #model_id='logistic'
#     #model_id='svc'
#     #model_id='RF'
#     #model_id='Ada'
    
#     if flag_ML==0:
#         if flag_sensitivity==1:
#             parameter = [11,12] #,13,14,15,16,17,18,19,20
#             for i in range(len(parameter)):
#                 avg1,avg2,res_all, res_all_F1=  run(model_id,n_steps_in,n_steps_out,n_features,
#                                parameter[i],n_trivals,n_out,n_n_lstm,dropout,bat_size)  
#                 accuracy_avg_1.append(avg1)
#                 accuracy_avg_2.append(avg2)            
#         else:   
#             avg1,avg2,res_all, res_all_F1, avg_metrics_prec_recall_F1=  run(model_id,n_steps_in,n_steps_out,n_features,
#                                 n_epoch_global,n_trivals,n_out,n_n_lstm,dropout,bat_size)  
#             accuracy_avg_1.append(avg1)
#             accuracy_avg_2.append(avg2) 
            
#         print('model: ', model_id)
#         print('sensitivity_flag = ', flag_sensitivity) 
#         if flag_sensitivity==1:
#             print('parameter : ', parameter) 
#         print('n_step out: ', n_steps_out)
#         print('n_epoch,n_trivals, n_n_lstm,dropout,bat_size', 
#               n_epoch_global,n_trivals,n_n_lstm,dropout,bat_size)       
#         print('accuracy_avg_1: ',accuracy_avg_1)
#         print('accuracy_avg_2: ',accuracy_avg_2)
#         print('avg_metrics_prec_recall_F1= ',avg_metrics_prec_recall_F1)
       
#     else:        
#         vec_mean_metrics = run_ML(model_id,n_steps_out)
        
#         mean_all  = np.mean(vec_mean_metrics,axis=0)
#         print('vec_mean_metrics',vec_mean_metrics)
#         print('_mean_all',mean_all)

# main() 

Epoch 1/15
5235/5235 - 50s - loss: 0.2484 - accuracy: 0.2079 - 50s/epoch - 10ms/step
Epoch 2/15
5235/5235 - 48s - loss: 0.2321 - accuracy: 0.2005 - 48s/epoch - 9ms/step
Epoch 3/15
5235/5235 - 52s - loss: 0.2254 - accuracy: 0.2148 - 52s/epoch - 10ms/step
Epoch 4/15
5235/5235 - 48s - loss: 0.2227 - accuracy: 0.2076 - 48s/epoch - 9ms/step
Epoch 5/15
5235/5235 - 47s - loss: 0.2212 - accuracy: 0.2028 - 47s/epoch - 9ms/step
Epoch 6/15
5235/5235 - 51s - loss: 0.2207 - accuracy: 0.2000 - 51s/epoch - 10ms/step
Epoch 7/15
5235/5235 - 56s - loss: 0.2202 - accuracy: 0.1989 - 56s/epoch - 11ms/step
Epoch 8/15
5235/5235 - 51s - loss: 0.2196 - accuracy: 0.1971 - 51s/epoch - 10ms/step
Epoch 9/15
5235/5235 - 49s - loss: 0.2191 - accuracy: 0.1968 - 49s/epoch - 9ms/step
Epoch 10/15
5235/5235 - 44s - loss: 0.2188 - accuracy: 0.1996 - 44s/epoch - 8ms/step
Epoch 11/15
5235/5235 - 44s - loss: 0.2191 - accuracy: 0.1990 - 44s/epoch - 8ms/step
Epoch 12/15
5235/5235 - 53s - loss: 0.2188 - accuracy: 0.1988 - 53s/e

****

In [3]:
# Function to split a multivariate sequence into input-output samples for LSTM training
def split_sequences(sequences, n_steps_in, n_steps_out):
    """
    Splits the multivariate time series dataset into input-output sequences for training.
    
    Parameters:
    - sequences: The dataset containing both input features and target variables.
    - n_steps_in: Number of time steps to use as input for the LSTM.
    - n_steps_out: Number of time steps to predict in the output.

    Returns:
    - X: Array of input sequences (features).
    - y: Array of output sequences (target values).
    """
    X, y = list(), list()  # Initialize empty lists to store input (X) and output (y) sequences.
    
    # Iterate through the dataset to generate input-output pairs
    for i in range(len(sequences)):
        # Determine the end of the input sequence
        end_ix = i + n_steps_in
        # Determine the end of the output sequence
        out_end_ix = end_ix + n_steps_out - 1
        
        # Stop if we go beyond the dataset length
        if out_end_ix > len(sequences):
            break
        
        # Extract input features (excluding the target 'y') and output (target 'y')
        seq_x = sequences[i:end_ix, :-1]    # Input includes columns 't', 'dayofweek', 'weekend', 'y_t_1'
        seq_y = sequences[end_ix-1:out_end_ix, -1]  # Output is the column 'y' (occupancy prediction)
        X.append(seq_x)
        y.append(seq_y)
        
    return array(X), array(y)

In [4]:
# Function to read and preprocess data for the hybrid LSTM model
def read_data(string, string2, model_id, n_steps_in, n_steps_out, n_features):
    """
    Reads, preprocesses and prepares data for training and testing the LSTM model.
    
    Parameters:
    - string: Path to the main dataset (occupancy data).
    - string2: Path to the contextual dataset (weekday/weekend profiles).
    - model_id: Identifier for the type of model to be used.
    - n_steps_in: Number of time steps to use as input.
    - n_steps_out: Number of time steps to predict as output.
    - n_features: Number of features to consider in the input.
    
    Returns:
    - X_train, y_train: Training input and output sequences.
    - X_test, y_test: Testing input and output sequences.
    - X2_train, X2_test: Training and testing contextual data (weekday/weekend profiles).
    """
    # Read the main dataset (occupancy data)
        # - 't': Time of the day in 10-minute intervals
        # - 'dayofweek': Numeric representation of the day of the week (0 = Monday, ..., 6 = Sunday)
        # - 'weekend': Indicator of whether the day is a weekend (0 = No, 1 = Yes)
        # - 'y_t_1': Occupancy state at the previous time step
        # - 'y': Occupancy state at the current time step (target variable)
    Z = pd.read_csv(string)
    Z = Z.to_numpy()
    
    # Split the dataset into input (X) and output (y) sequences
    X, y = split_sequences(Z[:,3:5], n_steps_in, n_steps_out) # Use `y_t_1` as input, `y` as target
    n_train = int(0.7 * len(X))  # Define 70% of the data for training
    
    # Read the contextual data (weekday/weekend profiles)
    Z1 = pd.read_csv(string2)
    Z1 = Z1.to_numpy()
    Z1 = Z1.transpose()  # Transpose to have weekday and weekend in the same dimension
    
    # Create a replicated version of the contextual data to align with the input size
    Z2 = np.concatenate((Z1, Z1), axis=1)  # Concatenate weekday and weekend profiles for each time step
    
    # Initialize a zero matrix for the combined feature set
    X2 = np.zeros([len(Z), 3 + 144], float)  # 3 original features (t, DayOfWeek, Weekend) + 144 contextual features (occupancy profile rates)
    
    # Loop through the data to populate the contextual features (weekday/weekend profiles)
    for i in range(len(Z) - n_steps_in):
        # Check if the current sample corresponds to a weekday or weekend
        if Z[i + n_steps_in - 1, -1] == 0:  # Check the 'weekend' column
            qq = np.array(Z2[0][0:144])  # Select weekday profile data
            X2[i] = np.append(Z[i + n_steps_in - 1][0:3], qq)  # Combine original features with contextual data

        else:  # For weekend
            qq = np.array(Z2[1][0:144])  # Select weekend profile data
            X2[i] = np.append(Z[i + n_steps_in - 1][0:3], qq)  # Combine original features with contextual data
    
    # Split dataset into training and testing sets
    X_train = X[0:n_train,]; y_train = y[0:n_train,]
    X_test = X[n_train:len(X),]; y_test  = y[n_train:len(X),]
    
    # Split contextual data into training and testing sets
    X2_train = X2[0:n_train,]; X2_test = X2[n_train:len(X),];  
    
    return X_train, y_train, X_test, y_test, X2_train, X2_test

In [5]:
# Function to define, train and evaluate the hybrid LSTM model
def fit_model_MixLSTM(res_F1, res, _iter, X_train, y_train, X_test, y_test, X2_train, X2_test,
                      n_steps_in, n_steps_out, n_features, n_n_lstm, dropout, n_epoch, bat_size):
    """
    Defines and trains the hybrid LSTM model that combines sequential data with contextual information.

    Parameters:
    - res_F1, res: Arrays to store the evaluation metrics (F1 score, accuracy, etc.).
    - _iter: The iteration index (useful for multiple trials).
    - X_train, y_train: Training data inputs and outputs.
    - X_test, y_test: Testing data inputs and outputs.
    - X2_train, X2_test: Contextual data inputs (weekday/weekend profiles).
    - n_steps_in, n_steps_out: Number of input and output steps for the LSTM.
    - n_features: Number of features in the input data.
    - n_n_lstm: Number of neurons in the LSTM layer.
    - dropout: Dropout rate to avoid overfitting.
    - n_epoch: Number of training epochs.
    - bat_size: Batch size for training.
    
    Returns:
    - Updated res_F1 and res with evaluation metrics.
    """
    
    # Input layer for sequential LSTM data
    input1 = keras.Input(shape=(n_steps_in, n_features))  # Input for the LSTM branch
    # Input layer for contextual dense data
    input2 = keras.Input(shape=(147,))  # Input for the dense layer branch 147 contextual features (3 core features and 144 contextual)
    
    # LSTM branch: Processes sequential input data ('y_t_1' as input and 'y' as output)
    model_LSTM = LSTM(n_n_lstm)(input1)  # LSTM layer with n_n_lstm neurons for capturing temporal dependencies
    model_LSTM = Dropout(dropout)(model_LSTM)  # Dropout layer to prevent overfitting
    model_LSTM = Dense(18, activation='relu')(model_LSTM)  # Dense layer to refine LSTM output, 18 neurons
    
    # Contextual data branch: Processes additional weekday/weekend profiles
    meta_layer = keras.layers.Dense(147, activation="relu")(input2)  # First dense layer for contextual data
    meta_layer = keras.layers.Dense(64, activation="relu")(meta_layer)  # Second dense layer with 64 neurons
    meta_layer = keras.layers.Dense(32, activation="relu")(meta_layer)  # Third dense layer with 32 neurons
    
    # Merging both branches (concatenating LSTM output and contextual data output)
    model_merge = keras.layers.concatenate([model_LSTM, meta_layer])  # Concatenate outputs from LSTM and dense layers
    model_merge = Dense(100, activation='relu')(model_merge)  # Further dense layer after merging
    model_merge = Dropout(dropout)(model_merge)  # Additional dropout layer
    
    # Output layer: Sigmoid activation function for binary classification (occupancy prediction)
    output = Dense(n_steps_out, activation='sigmoid')(model_merge)
    
    # Compile the model using binary crossentropy loss and the Adam optimizer
    model = Model(inputs=[input1, input2], outputs=output) 
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    keras.utils.plot_model(model, show_shapes=True)  # Plot the model architecture
    
    # Train the model using the training data
    model.fit([X_train, X2_train], y_train, epochs=n_epoch, batch_size=bat_size, verbose=2)
    
    # Predict using the test data
    temp = model.predict([X_test, X2_test], verbose=2)
    m, n = temp.shape 
    t_target = n_steps_out
       
    yhat = np.zeros((m, t_target))  # Initialize matrix for predicted values
    y_obs = np.array(y_test[0:m, 0:t_target])  # Get observed values for comparison
    scores1 = np.zeros(m)
    scores_F1 = np.zeros([m, 3], float)  # Initialize matrix to store F1 scores

    # Evaluate the predictions and calculate scores
    for i in np.arange(m):  
        for j in np.arange(t_target):  
            if temp[i][j] >= 0.5:
                yhat[i][j] = 1  # Convert probabilities to binary predictions
        val = 1 - sum(abs(yhat[i,] - y_obs[i,:])) / t_target  # Calculate prediction accuracy
        # Compute precision, recall, and F1 score
        scores_F1[i, 0] = precision_score(y_obs[i,:], yhat[i,], zero_division=1)
        scores_F1[i, 1] = recall_score(y_obs[i,:], yhat[i,], zero_division=1)
        scores_F1[i, 2] = f1_score(y_obs[i,:], yhat[i,], zero_division=1)
        scores1[i] = val  # Store accuracy score
     
    _mean1 = np.mean(scores1)  # Mean accuracy score
    _mean_F1 = np.mean(scores_F1, axis=0)  # Mean F1 score for each metric
    res[_iter,:] = [n_n_lstm, dropout, n_epoch, bat_size, _mean1]  # Store results for each iteration
    res_F1[_iter,:] = _mean_F1
    return res_F1, res     

In [6]:
# Main function to execute the entire process
def run(model_id, n_steps_in, n_steps_out, n_features, n_epoch, n_trivals, n_out,
        n_n_lstm, dropout, bat_size): 
     
    t_win = n_steps_in * n_steps_out  # Total input-output window size        
    n_station = 27  # Number of stations to process
    
    # Define the paths to the datasets
    string = '../Datasets/occupancy_data/ChargingData/data_chg_'
    string2 = '../Datasets/occupancy_data/RateOfChargingData/data_chg_pred_occ_t_'
    station = [string + f'{i}.csv' for i in range(1, 28)]
    station2 = [string2 + f'{i}.csv' for i in range(1, 28)]
  
    res_all = []; res_all_F1 = []  # Initialize lists to store results
    # Iterate over each station
    for s in range(n_station):         
        # Read data for the current station
        X_train, y_train, X_test, y_test, X2_train, X2_test = read_data(station[s], station2[s], model_id,  
                                n_steps_in, n_steps_out, n_features)
        res = np.zeros([n_trivals, n_out])  # Initialize result matrices
        res_F1 = np.zeros([n_trivals, 3])
        
        # Iterate over the number of trials to train the model
        for _iter in range(n_trivals):   
            if model_id == 'Mix_LSTM':
                # Fit the model and get the evaluation results
                res_F1, res = fit_model_MixLSTM(res_F1, res, _iter, X_train, y_train, X_test, y_test, X2_train, X2_test, 
                           n_steps_in, n_steps_out, n_features, n_n_lstm, dropout, n_epoch,
                           bat_size)  
               
        _mean = np.mean(res[:,-1:], axis=0)  # Calculate mean accuracy
        _std  = np.std(res[:,-1:], axis=0)  # Calculate standard deviation of accuracy
        res_all.append([_mean, _std])  # Append results to the list
       
        _mean_F1 = np.mean(res_F1, axis=0)  # Mean F1 scores
        _std_F1  = np.std(res_F1, axis=0)  # Standard deviation of F1 scores
        res_all_F1.append([_mean_F1])
        
    temp = []; temp1 = []
    # Calculate overall averages
    for i in range(n_station):            
        temp.append(res_all[i][0])     
        
    accuracy_avg1 = np.mean(temp, axis=0)  # Mean accuracy over stations
    accuracy_avg2 = np.mean(temp, axis=1)  # Mean accuracy over trials
    avg_metrics_prec_recall_F1 = np.mean(res_all_F1, axis=0)  # Mean precision, recall, and F1 scores
    
    # Return all computed metrics
    return accuracy_avg1, accuracy_avg2, res_all, res_all_F1, avg_metrics_prec_recall_F1

In [7]:
from tabulate import tabulate
import time

In [ ]:
# Main function to set parameters and run the model
def main():
    # Start time measurement
    start_time = time.time()


    # Define hyperparameters and model settings
    n_steps_in = 12  # Number of input steps for LSTM
    n_features = 1  # Number of features for LSTM input
    n_steps_out = 6  # Number of output steps to predict
    n_epoch_global = 15  # Number of epochs for training
    n_trivals = 10  # Number of trials to run
    n_out = 5  # Output dimension
    n_n_lstm = 36  # Number of neurons in LSTM layer
    dropout = 0.2  # Dropout rate
    bat_size = 30  # Batch size
    model_id = 'Mix_LSTM'  # Model identifier for conditional logic

    # Initialize list to store accuracy results
    accuracy_avg_1 = [] 
    accuracy_avg_2 = []
    
    # Run the model with the defined parameters
    avg1, avg2, res_all, res_all_F1, avg_metrics_prec_recall_F1 = run(model_id, n_steps_in, n_steps_out, n_features,
                                                                    n_epoch_global, n_trivals, n_out, n_n_lstm, dropout, bat_size)  
    # Append results to the lists
    accuracy_avg_1.append(avg1)
    accuracy_avg_2.append(avg2) 
            
    # Print out the results
    print("\n================= Model Training Summary =================")
    print(f"Model: {model_id}")
    print(f"Number of Predicted Steps: {n_steps_out}\n")
    
    print("Hyperparameters:")
    print(f"- Number of Epochs: {n_epoch_global}")
    print(f"- Number of Trials: {n_trivals}")
    print(f"- Number of Neurons in LSTM Layer: {n_n_lstm}")
    print(f"- Dropout Rate: {dropout}")
    print(f"- Batch Size: {bat_size}\n")

    print("Average Accuracy Scores:")
    print(f"- Accuracy (Average 1): {accuracy_avg_1}")
    print(f"- Accuracy (Average 2): {accuracy_avg_2}\n")

    # Use tabulate to display metrics in a formatted table
    headers = ["Metric", "Score"]
    metrics_table = [
        ["Precision", f"{avg_metrics_prec_recall_F1[0]:.4f}"],
        ["Recall", f"{avg_metrics_prec_recall_F1[1]:.4f}"],
        ["F1 Score", f"{avg_metrics_prec_recall_F1[2]:.4f}"]
    ]
    print("Average Metrics (Precision, Recall, F1 Score):")
    print(tabulate(metrics_table, headers=headers, tablefmt="pretty"))
    
    print("==========================================================")

    # End time measurement
    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"\nTotal execution time: {elapsed_time:.2f} seconds")
        
main()